# Lab 07 — Frameworks vs. Plain Code

Kiel University · Agentic AI (infAgAI-01a) · Winter 2026

**Learning objectives** — after this lab you can:

- name the **five framework families** and state what each believes an agent is,
- list what a framework **buys** (the six services) and what it **costs** (the deferred column),
- grow the S02 loop into the lecture's **~100-line plain baseline**: tool registry, retry policy with exponential backoff, JSON checkpointing, resume,
- rebuild **LangGraph's API shape in plain Python** — typed state with an `add_messages` reducer, nodes, conditional edges, a SQLite checkpointer with `thread_id` — and run the same research agent on it,
- run the lecture's **four measurements**: lines of code, dependency count, time to diagnose an injected description-mismatch bug, and the kill test,
- apply the **decision checklist** and the **design-for-divorce** exit practices to a concrete recommendation (the CTO memo).

> ⏱️ Estimated time: 90–120 minutes. The two live agent runs (Parts C and D) take a few
> minutes each on a local model — start them early and read on while they run.

## Theory recap — the build-or-adopt decision

### The question, and the landscape

Session 06 chose the orchestration pattern; this session chooses **who implements it**. The
one idea to keep in view: *a framework is code you run but did not write — including prompts.*
Five families are on offer, each defined by what it believes an agent is. **LangGraph**: an
explicit **state graph** — typed shared state, nodes that return updates, conditional edges
that route on the model's output, checkpointing built in. **CrewAI**: a **team of
role-players** — agents with role, goal and backstory, run sequentially or under a manager;
fast to demo, heavily opinionated. **AutoGen**: everything is a **conversation** — control
flow *emerges* from the dialogue. **smolagents**: minimalism, ~1,000 core lines — agents act
by **writing Python code** rather than JSON tool calls (the CodeAct result, Wang et al. 2024).
**Vendor SDKs** (OpenAI Agents SDK: agents, handoffs, guardrails, sessions; Claude Agent SDK:
Claude Code's harness with hooks, subagents, permissions): thin, provider-tuned — and also a
distribution channel. All five wrap the same loop from Session 02; they differ in **what they
make explicit** and what they hide.

### The ledger: what you buy, what you pay

A framework sells **six services**: *state management* (typed schemas with merge rules),
*retries & error policy* (declared backoff instead of copy-pasted try/except), *persistence*
(checkpoint after every step — resume, rewind-and-replay), *tracing hooks*, *streaming
plumbing*, and *human-in-the-loop* gates (persistence wearing a safety hat). The costs are
equally real but **deferred**: *abstraction opacity* (prompts you never read execute in your
name), *debugging through layers*, *version churn* (LangChain went 0.1 → 1.0 in under two
years), *lock-in* (state schemas and checkpointer formats become load-bearing), and *hidden
prompt magic* — your agent's behaviour can change on `pip install -U` without you editing a
single line. The **timing asymmetry** is the analytical heart: benefits arrive on day one, at
the demo; costs arrive in month three, in production. Price both on the same timescale.

### Start simple — and the 100-line baseline

Anthropic's *Building Effective Agents* (2024): find the simplest solution possible; add
complexity only when it **demonstrably** improves outcomes; use LLM APIs directly first; if
you adopt a framework, understand what is under the hood. Hence the **maturity inversion**:
beginners reach for frameworks to avoid the loop; experienced teams write the loop first. The
lecture's baseline is ~100 honest lines: a **tool registry** (a plain dict), an eight-line
**retry wrapper** sleeping $2^k$ seconds (1 s, 2 s, 4 s), a two-line **JSON checkpoint** saved
after *every* step, and a **resume** function. What it does *not* give you: checkpoint
history with replay, thread management for concurrent runs, suspension-based approval gates.
The LangGraph version of the same agent is ~40 lines — but **complexity is conserved**: the
missing lines moved into a dependency, where they still execute and are read by no one. The
difference is **ownership**, not behaviour. Whichever way you decide: **design for divorce** —
ports and adapters, own every prompt, own the message log (plain provider-format messages are
the portable core of any agent), and keep an eval suite as the exit door.

### This lab

Exactly what the lecture announced: **build it both ways**. The task is the course's research
agent — search, evaluate sources, draft a Markdown report. **Variant A**: your S02 loop grown
to ~100 lines with a registry, retries and JSON checkpointing. **Variant B**: the same
behaviour as a state graph. The lecture names **LangGraph**; true to the course's
build-it-yourself-first stance, you will not install it — you will **mirror its API shape in
plain Python** (a *minigraph*: `add_messages` reducer, `ToolNode`, `tools_condition`, a
`SqliteSaver` on stdlib `sqlite3`), so every line of the "framework" is also yours to read.
Web search runs against a tiny **offline corpus** of saved pages in `data/` (no live web
access needed). Then you **measure rather than opine** — LOC, dependencies, an injected
description-mismatch bug, the kill test — and write the memo (R3).

## Part A — Setup & Ollama connectivity

One tool-capable local model is enough for this lab (default `qwen2.5:7b`, override with the
environment variable `OLLAMA_MODEL`). Every LLM cell degrades gracefully if Ollama is not
reachable — all framework-mechanics cells (Parts B, D and E) run without a model.

In [ ]:
import os
import json
import time
import sqlite3
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")   # any tool-capable 7-30B model works
OLLAMA_OK = False
AVAILABLE = []


def _field(obj, name, default=None):
    """Read a field from dict-like or attribute-style Ollama responses (client versions differ)."""
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)


try:
    import ollama
    listing = ollama.list()
    for m in _field(listing, "models", []) or []:
        name = _field(m, "model", None) or _field(m, "name", "")
        if name:
            AVAILABLE.append(name)
    OLLAMA_OK = True
    print("Ollama is running. Local models:", ", ".join(AVAILABLE) or "(none)")
    base = MODEL.split(":")[0]
    if not any(a == MODEL or a.split(":")[0] == base for a in AVAILABLE):
        print(f"Model {MODEL!r} not found — run `ollama pull {MODEL}`.")
except Exception as exc:
    print("Could not reach Ollama:", exc)
    print("→ Start Ollama with `ollama serve` and pull the model with `ollama pull qwen2.5:7b`.")
    print("  The framework cells below still run; only the live agent runs are skipped.")

> **Q:** Each framework family "believes" an agent is something different. State the core abstraction of LangGraph, CrewAI, AutoGen and smolagents in one sentence each.
<details><summary>Click for answer</summary>

LangGraph models an agent as an explicit <em>state graph</em>: typed shared state transformed
by nodes connected with (conditional) edges. CrewAI models an agent system as a <em>team of
role-players</em>: agents defined by role, goal and backstory, executing tasks sequentially or
under a manager. AutoGen models everything as a <em>conversation</em>: agents (and humans)
exchange messages, and control flow emerges from the dialogue. smolagents models the agent as
a <em>code writer</em>: actions are model-authored Python snippets executed in a sandbox,
rather than JSON tool calls.
</details>

## Part B — The research task, its tools, and the registry

The running example stays the course's **research agent**: brief → search → evaluate sources →
draft a Markdown report. Because the lab needs no live web access, `web_search` runs over a
tiny **offline corpus of eight saved pages** in `data/corpus.json` (synthetic, but each page
mirrors a source type you met in the lecture: a vendor essay, framework docs, engineering war
stories, a marketing post, a forum thread, a survey).

Exactly as on the lecture's plain-code slide, the agent has **two tools** in a plain-dict
**tool registry**: `web_search` and `save_report`. Note two deliberate design decisions that
you will meet again in the exit-strategy discussion: the **system prompt is owned** — it lives
in this notebook, not in any library — and tool observations flow back as plain
**provider-format messages**.

In [ ]:
CORPUS_PATH = "data/corpus.json"
corpus = json.loads(Path(CORPUS_PATH).read_text(encoding="utf-8"))
print(f"{len(corpus)} saved pages in the offline corpus:",
      ", ".join(d["id"] for d in corpus))


def score(doc, terms):
    """Keyword relevance of one document: hits in the title count double."""
    title, text = doc["title"].lower(), doc["text"].lower()
    return sum(2 * (t in ___) + (t in text) for t in terms)


def web_search(query: str) -> str:
    """Search the saved research corpus; return the three MOST relevant sources."""
    terms = [t for t in query.lower().split() if len(t) > 2]
    ranked = sorted(corpus, key=lambda d: score(d, terms), reverse=___)
    hits = [d for d in ranked if score(d, terms) > 0][:3]
    if not hits:
        return "No results for this query. Try different terms."
    return "\n\n".join(f"[{d['id']}] {d['title']} ({d['source_type']}, {d['date']})\n"
                       f"URL: {d['url']}\n{d['text']}" for d in hits)


def save_report(filename: str, content: str) -> str:
    """Save a Markdown report to the reports/ folder; returns a confirmation."""
    outdir = Path("reports")
    outdir.mkdir(exist_ok=True)
    out = outdir / Path(filename).name
    out.write_text(content, encoding="utf-8")
    return f"Report saved to {out} ({len(content)} characters)."


TOOLS = {"web_search": ___, "save_report": save_report}   # the registry: a plain dict

print("\n--- smoke test ---")
print(web_search("agent framework checkpointing")[:350], "…")

<details>
<summary><b>Click here for the solution</b></summary>

```python
CORPUS_PATH = "data/corpus.json"
corpus = json.loads(Path(CORPUS_PATH).read_text(encoding="utf-8"))
print(f"{len(corpus)} saved pages in the offline corpus:",
      ", ".join(d["id"] for d in corpus))


def score(doc, terms):
    """Keyword relevance of one document: hits in the title count double."""
    title, text = doc["title"].lower(), doc["text"].lower()
    return sum(2 * (t in title) + (t in text) for t in terms)


def web_search(query: str) -> str:
    """Search the saved research corpus; return the three MOST relevant sources."""
    terms = [t for t in query.lower().split() if len(t) > 2]
    ranked = sorted(corpus, key=lambda d: score(d, terms), reverse=True)
    hits = [d for d in ranked if score(d, terms) > 0][:3]
    if not hits:
        return "No results for this query. Try different terms."
    return "\n\n".join(f"[{d['id']}] {d['title']} ({d['source_type']}, {d['date']})\n"
                       f"URL: {d['url']}\n{d['text']}" for d in hits)


def save_report(filename: str, content: str) -> str:
    """Save a Markdown report to the reports/ folder; returns a confirmation."""
    outdir = Path("reports")
    outdir.mkdir(exist_ok=True)
    out = outdir / Path(filename).name
    out.write_text(content, encoding="utf-8")
    return f"Report saved to {out} ({len(content)} characters)."


TOOLS = {"web_search": web_search, "save_report": save_report}   # the registry: a plain dict

print("\n--- smoke test ---")
print(web_search("agent framework checkpointing")[:350], "…")
```

</details>

In [ ]:
SCHEMAS = [
    {"type": "function", "function": {
        "name": "web_search",
        "description": ("Search the saved research corpus. Returns the three most relevant "
                        "sources with id, title, URL, date and a text snippet."),
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string",
                                                "description": "search terms"}},
                       "required": [___]}}},
    {"type": "function", "function": {
        "name": "save_report",
        "description": "Save the final Markdown report to disk.",
        "parameters": {"type": "object",
                       "properties": {"filename": {"type": "string"},
                                      "content": {"type": "string",
                                                  "description": "Markdown text"}},
                       "required": ["filename", "content"]}}},
]


def to_dict(msg):
    """Normalise an Ollama message (object or dict) into a plain, JSON-serialisable dict."""
    if hasattr(msg, "model_dump"):
        msg = msg.model_dump(exclude_none=True)
    keep = {k: msg[k] for k in ("role", "content", "tool_calls") if msg.get(k) is not None}
    keep.setdefault("content", "")
    return json.loads(json.dumps(keep, default=str))   # round-trip: guaranteed serialisable


def run_tool(call, registry=None):
    """Execute ONE tool call; wrap the observation as a role='tool' message (S03 convention:
    tool errors are observations, not crashes)."""
    registry = registry if registry is not None else TOOLS
    fn = call["function"]["name"]
    args = call["function"]["arguments"]
    if isinstance(args, str):
        args = json.loads(___)
    try:
        result = registry[___](**args)
    except Exception as exc:
        result = f"Tool error: {exc}"
    return {"role": "tool", "tool_name": fn, "content": str(result)}


# The prompt is OURS: it lives in this notebook, version-controlled — never a library default.
SYSTEM_PROMPT = (
    "You are a careful research agent. Work step by step: search the corpus with web_search "
    "(vary your queries and search at least twice), weigh the sources (prefer framework docs, "
    "surveys and engineering war stories over marketing), then write a concise Markdown "
    "report — title, a clear recommendation, and a Sources section citing the URLs you used — "
    "and save it with save_report. Only after the report is saved, answer with ONE plain-text "
    "sentence summarising the recommendation, without any tool call."
)


def init_messages(task):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": task}]


TASK = ("Research question: should a two-person startup building its first LLM agent adopt "
        "an agent framework or write plain Python? Investigate with the tools, then save "
        "your findings as 'framework_memo.md'.")

print("--- smoke test (no LLM involved) ---")
probe_call = {"function": {"name": "web_search",
                           "arguments": {"query": "hidden prompts dependency update"}}}
print(run_tool(probe_call)["content"][:300], "…")

<details>
<summary><b>Click here for the solution</b></summary>

```python
SCHEMAS = [
    {"type": "function", "function": {
        "name": "web_search",
        "description": ("Search the saved research corpus. Returns the three most relevant "
                        "sources with id, title, URL, date and a text snippet."),
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string",
                                                "description": "search terms"}},
                       "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "save_report",
        "description": "Save the final Markdown report to disk.",
        "parameters": {"type": "object",
                       "properties": {"filename": {"type": "string"},
                                      "content": {"type": "string",
                                                  "description": "Markdown text"}},
                       "required": ["filename", "content"]}}},
]


def to_dict(msg):
    """Normalise an Ollama message (object or dict) into a plain, JSON-serialisable dict."""
    if hasattr(msg, "model_dump"):
        msg = msg.model_dump(exclude_none=True)
    keep = {k: msg[k] for k in ("role", "content", "tool_calls") if msg.get(k) is not None}
    keep.setdefault("content", "")
    return json.loads(json.dumps(keep, default=str))   # round-trip: guaranteed serialisable


def run_tool(call, registry=None):
    """Execute ONE tool call; wrap the observation as a role='tool' message (S03 convention:
    tool errors are observations, not crashes)."""
    registry = registry if registry is not None else TOOLS
    fn = call["function"]["name"]
    args = call["function"]["arguments"]
    if isinstance(args, str):
        args = json.loads(args)
    try:
        result = registry[fn](**args)
    except Exception as exc:
        result = f"Tool error: {exc}"
    return {"role": "tool", "tool_name": fn, "content": str(result)}


# The prompt is OURS: it lives in this notebook, version-controlled — never a library default.
SYSTEM_PROMPT = (
    "You are a careful research agent. Work step by step: search the corpus with web_search "
    "(vary your queries and search at least twice), weigh the sources (prefer framework docs, "
    "surveys and engineering war stories over marketing), then write a concise Markdown "
    "report — title, a clear recommendation, and a Sources section citing the URLs you used — "
    "and save it with save_report. Only after the report is saved, answer with ONE plain-text "
    "sentence summarising the recommendation, without any tool call."
)


def init_messages(task):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": task}]


TASK = ("Research question: should a two-person startup building its first LLM agent adopt "
        "an agent framework or write plain Python? Investigate with the tools, then save "
        "your findings as 'framework_memo.md'.")

print("--- smoke test (no LLM involved) ---")
probe_call = {"function": {"name": "web_search",
                           "arguments": {"query": "hidden prompts dependency update"}}}
print(run_tool(probe_call)["content"][:300], "…")
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `SCHEMAS` is what the *model* reads about the tools; `TOOLS` is what actually runs.
  Keeping the two side by side in one place is Variant A's quiet advantage — in Part E we
  will inject a bug exactly into the gap between them.
- `to_dict` matters more than it looks: the whole lab persists state as **plain
  provider-format message dicts** ("own the message log"). The `json` round-trip guarantees
  every checkpoint is serialisable, whatever object type the Ollama client returns.
- `run_tool` takes an optional `registry` so the minigraph's `ToolNode` (Part D) and the
  buggy registry (Part E) can reuse it unchanged.

</details>

> **Q:** Name the six framework-provided services discussed in the lecture.
<details><summary>Click for answer</summary>

State management (typed schemas with merge rules), retries and error policy (declarative
backoff, timeouts, fallbacks), persistence/checkpointing (durable resume, replay), tracing
hooks (structured spans per LLM/tool call), streaming plumbing (token/event propagation to
UIs), and human-in-the-loop primitives (interrupt and resume around risky nodes).
</details>

## Part C — Variant A: the plain-code baseline (~100 lines)

The S02 loop, grown up exactly as on the lecture's two plain-code slides: an explicit **retry
policy** (exponential backoff, $2^k$ seconds), and the cheapest persistence layer you will
ever meet — **serialise the full state to JSON after every step**, plus a `resume` function.
The state is a dict holding the message list (the agent's complete working memory, S02) and a
step counter, so even the position in the step budget survives a crash.

`SimulatedCrash` and `CRASH_BUDGET` stand in for `kill -9`: Part E arms them to kill the
process mid-run; until then they are dormant.

In [ ]:
class SimulatedCrash(RuntimeError):
    """Stands in for kill -9 / a power failure in Part E's kill test."""


CRASH_BUDGET = {"calls_left": None}          # None = never crash (Part E arms this)

OPTIONS = {"temperature": 0.2}               # sampling options, tweakable in Part F


def call_llm(messages, retries=3):
    """One LLM step with an explicit retry policy: exponential backoff 1 s, 2 s, 4 s."""
    if CRASH_BUDGET["calls_left"] is not None:
        if CRASH_BUDGET["calls_left"] <= 0:
            raise SimulatedCrash("process killed mid-run (simulated kill -9)")
        CRASH_BUDGET["calls_left"] -= 1
    for attempt in range(retries):
        try:
            resp = ollama.chat(model=MODEL, messages=messages, tools=SCHEMAS,
                               options=OPTIONS)
            return to_dict(resp[___])
        except Exception:
            time.sleep(___ ** attempt)         # 1 s, 2 s, 4 s
    raise RuntimeError("LLM down; state was saved")


def save(state, path="run.json"):            # checkpointing, two honest lines
    Path(path).write_text(json.dumps(___), encoding="utf-8")


def resume(path="run.json"):                 # ...and the ten-line resume is mostly json.loads
    return json.loads(Path(path).read_text(encoding="utf-8"))


print("retry policy: 3 attempts, sleeping", [2 ** a for a in range(3)], "seconds between them")

<details>
<summary><b>Click here for the solution</b></summary>

```python
class SimulatedCrash(RuntimeError):
    """Stands in for kill -9 / a power failure in Part E's kill test."""


CRASH_BUDGET = {"calls_left": None}          # None = never crash (Part E arms this)

OPTIONS = {"temperature": 0.2}               # sampling options, tweakable in Part F


def call_llm(messages, retries=3):
    """One LLM step with an explicit retry policy: exponential backoff 1 s, 2 s, 4 s."""
    if CRASH_BUDGET["calls_left"] is not None:
        if CRASH_BUDGET["calls_left"] <= 0:
            raise SimulatedCrash("process killed mid-run (simulated kill -9)")
        CRASH_BUDGET["calls_left"] -= 1
    for attempt in range(retries):
        try:
            resp = ollama.chat(model=MODEL, messages=messages, tools=SCHEMAS,
                               options=OPTIONS)
            return to_dict(resp["message"])
        except Exception:
            time.sleep(2 ** attempt)         # 1 s, 2 s, 4 s
    raise RuntimeError("LLM down; state was saved")


def save(state, path="run.json"):            # checkpointing, two honest lines
    Path(path).write_text(json.dumps(state), encoding="utf-8")


def resume(path="run.json"):                 # ...and the ten-line resume is mostly json.loads
    return json.loads(Path(path).read_text(encoding="utf-8"))


print("retry policy: 3 attempts, sleeping", [2 ** a for a in range(3)], "seconds between them")
```

</details>

In [ ]:
MAX_STEPS = 12


def run_agent(task=None, state=None, checkpoint="run.json", verbose=True):
    """Variant A: the S02 loop grown up — retries, per-step JSON checkpointing, resume.
    Pass `state=` (from resume()) to continue an interrupted run."""
    if state is None:
        state = {"messages": init_messages(task), "step": 0}
    while state["step"] < ___:
        msg = call_llm(state["messages"])
        state["messages"].append(msg)
        state["step"] += 1
        save(state, checkpoint)              # survive a crash, resume later
        if not msg.get(___):        # plain text = final answer (S02 stop condition)
            if verbose:
                print("FINAL:", msg["content"])
            return state
        for call in msg["tool_calls"]:
            if verbose:
                print(f"step {state['step']:>2}: tool call → {call['function']['name']}")
            state["messages"].append(run_tool(___))
        save(state, checkpoint)              # ...and again after the observations
    print("Step budget exhausted without a final answer.")
    return state


if OLLAMA_OK:
    state_a = run_agent(TASK)
    print(f"\nVariant A finished after {state_a['step']} reasoning steps; "
          f"{len(state_a['messages'])} messages in the log.")
else:
    print("Ollama not reachable — skipping the live run (see Part A).")

<details>
<summary><b>Click here for the solution</b></summary>

```python
MAX_STEPS = 12


def run_agent(task=None, state=None, checkpoint="run.json", verbose=True):
    """Variant A: the S02 loop grown up — retries, per-step JSON checkpointing, resume.
    Pass `state=` (from resume()) to continue an interrupted run."""
    if state is None:
        state = {"messages": init_messages(task), "step": 0}
    while state["step"] < MAX_STEPS:
        msg = call_llm(state["messages"])
        state["messages"].append(msg)
        state["step"] += 1
        save(state, checkpoint)              # survive a crash, resume later
        if not msg.get("tool_calls"):        # plain text = final answer (S02 stop condition)
            if verbose:
                print("FINAL:", msg["content"])
            return state
        for call in msg["tool_calls"]:
            if verbose:
                print(f"step {state['step']:>2}: tool call → {call['function']['name']}")
            state["messages"].append(run_tool(call))
        save(state, checkpoint)              # ...and again after the observations
    print("Step budget exhausted without a final answer.")
    return state


if OLLAMA_OK:
    state_a = run_agent(TASK)
    print(f"\nVariant A finished after {state_a['step']} reasoning steps; "
          f"{len(state_a['messages'])} messages in the log.")
else:
    print("Ollama not reachable — skipping the live run (see Part A).")
```

</details>

> **Q:** Why is placing `save(state)` *inside* the loop, after every reasoning step, essential rather than cosmetic?
<details><summary>Click for answer</summary>

The checkpoint's value is bounded by its staleness: saved every step, a crash at any moment
loses at most the current step's work, and resume is seamless because the stateless model
never observes the interruption. Saved only at the end (or start), a crash mid-run loses
everything, and "persistence" is fiction. The placement decision is the difference between
durable execution and a souvenir file — analogous to the checkpointer running after every
node in LangGraph.
</details>

## Part D — Variant B: the minigraph (LangGraph's API shape, in plain Python)

The lecture built Variant B in **LangGraph**. True to the course stance — *build it yourself
first* — you will not `pip install` anything: you now write a **minigraph**, ~60 lines of
plain Python that mirror LangGraph's API shape one concept at a time:

| LangGraph | our mirror |
|---|---|
| `Annotated[list, add_messages]` reducer | `add_messages(existing, update)` + a `reducers` dict |
| `StateGraph`, `add_node`, `add_edge`, `add_conditional_edges`, `compile()` | `MiniGraph` with the same methods |
| `langgraph.prebuilt.ToolNode` / `tools_condition` | `ToolNode` class / `tools_condition` function |
| `SqliteSaver` checkpointer, `thread_id`, `recursion_limit` | `SqliteSaver` on stdlib `sqlite3`, same config keys |

Two payoffs over Variant A's snapshot, exactly the ones the lecture priced: the checkpointer
keeps a **full per-thread history** (enabling rewind-and-replay, task R2), and **resuming is a
config detail** — invoke the same `thread_id` again. And one honest difference to keep in
mind for the report: here *you* own every framework line; with real LangGraph those lines
live in a dependency.

In [ ]:
START, END = "__start__", "__end__"


def add_messages(existing, update):
    """Reducer: new messages are APPENDED, never overwrite (mirrors langgraph add_messages).
    The plain loop's implicit 'we always call append' convention, made a declared rule."""
    return list(___) + list(update)


def tools_condition(state):
    """Conditional edge (mirrors langgraph.prebuilt): route on the model's last message —
    tool calls go to the tools node, plain text ends the run."""
    last = state["messages"][-1]
    return ___ if last.get("tool_calls") else END


class ToolNode:
    """Prebuilt tool node (mirrors langgraph.prebuilt.ToolNode): executes every tool call
    in the last message and returns the observations as a state update."""

    def __init__(self, registry):
        self.registry = registry

    def __call__(self, state):
        last = state["messages"][-1]
        return {"messages": [run_tool(c, ___) for c in last.get("tool_calls", [])]}


class SqliteSaver:
    """Checkpointer (mirrors langgraph SqliteSaver): persists the FULL state after every
    node, keyed by thread_id — on nothing but stdlib sqlite3."""

    def __init__(self, path):
        self.con = sqlite3.connect(path)
        self.con.execute("CREATE TABLE IF NOT EXISTS checkpoints ("
                         "thread_id TEXT, step INTEGER, node TEXT, state TEXT, "
                         "PRIMARY KEY (thread_id, step))")

    def put(self, thread_id, step, node, state):
        self.con.execute("INSERT OR REPLACE INTO checkpoints VALUES (?, ?, ?, ?)",
                         (thread_id, step, node, json.dumps(___)))
        self.con.commit()

    def latest(self, thread_id):
        row = self.con.execute(
            "SELECT step, node, state FROM checkpoints WHERE thread_id = ? "
            "ORDER BY step DESC LIMIT 1", (thread_id,)).fetchone()
        if row is None:
            return None
        return {"step": row[0], "node": row[1], "state": json.loads(row[2])}

    def history(self, thread_id):
        rows = self.con.execute(
            "SELECT step, node, state FROM checkpoints WHERE thread_id = ? "
            "ORDER BY step", (thread_id,)).fetchall()
        return [{"step": s, "node": n, "state": json.loads(st)} for s, n, st in rows]

    def delete_after(self, thread_id, step):
        self.con.execute("DELETE FROM checkpoints WHERE thread_id = ? AND step > ?",
                         (thread_id, step))
        self.con.commit()


print("minigraph building blocks defined: add_messages, tools_condition, ToolNode, SqliteSaver")

<details>
<summary><b>Click here for the solution</b></summary>

```python
START, END = "__start__", "__end__"


def add_messages(existing, update):
    """Reducer: new messages are APPENDED, never overwrite (mirrors langgraph add_messages).
    The plain loop's implicit 'we always call append' convention, made a declared rule."""
    return list(existing) + list(update)


def tools_condition(state):
    """Conditional edge (mirrors langgraph.prebuilt): route on the model's last message —
    tool calls go to the tools node, plain text ends the run."""
    last = state["messages"][-1]
    return "tools" if last.get("tool_calls") else END


class ToolNode:
    """Prebuilt tool node (mirrors langgraph.prebuilt.ToolNode): executes every tool call
    in the last message and returns the observations as a state update."""

    def __init__(self, registry):
        self.registry = registry

    def __call__(self, state):
        last = state["messages"][-1]
        return {"messages": [run_tool(c, self.registry) for c in last.get("tool_calls", [])]}


class SqliteSaver:
    """Checkpointer (mirrors langgraph SqliteSaver): persists the FULL state after every
    node, keyed by thread_id — on nothing but stdlib sqlite3."""

    def __init__(self, path):
        self.con = sqlite3.connect(path)
        self.con.execute("CREATE TABLE IF NOT EXISTS checkpoints ("
                         "thread_id TEXT, step INTEGER, node TEXT, state TEXT, "
                         "PRIMARY KEY (thread_id, step))")

    def put(self, thread_id, step, node, state):
        self.con.execute("INSERT OR REPLACE INTO checkpoints VALUES (?, ?, ?, ?)",
                         (thread_id, step, node, json.dumps(state)))
        self.con.commit()

    def latest(self, thread_id):
        row = self.con.execute(
            "SELECT step, node, state FROM checkpoints WHERE thread_id = ? "
            "ORDER BY step DESC LIMIT 1", (thread_id,)).fetchone()
        if row is None:
            return None
        return {"step": row[0], "node": row[1], "state": json.loads(row[2])}

    def history(self, thread_id):
        rows = self.con.execute(
            "SELECT step, node, state FROM checkpoints WHERE thread_id = ? "
            "ORDER BY step", (thread_id,)).fetchall()
        return [{"step": s, "node": n, "state": json.loads(st)} for s, n, st in rows]

    def delete_after(self, thread_id, step):
        self.con.execute("DELETE FROM checkpoints WHERE thread_id = ? AND step > ?",
                         (thread_id, step))
        self.con.commit()


print("minigraph building blocks defined: add_messages, tools_condition, ToolNode, SqliteSaver")
```

</details>

In [ ]:
class MiniGraph:
    """Mirror of LangGraph's StateGraph surface: nodes, edges, conditional edges, compile()."""

    def __init__(self, reducers):
        self.reducers = reducers                 # field name -> merge rule
        self.nodes, self.edges, self.branches = {}, {}, {}

    def add_node(self, name, fn):
        self.nodes[name] = fn

    def add_edge(self, src, dst):
        self.edges[src] = dst

    def add_conditional_edges(self, src, router):
        self.branches[src] = router

    def compile(self, checkpointer=None):
        return CompiledGraph(self, checkpointer)


class CompiledGraph:
    """The runtime: walks the graph, merges node updates via reducers, checkpoints
    after EVERY node, and resumes any thread_id that already has checkpoints."""

    def __init__(self, graph, checkpointer):
        self.g, self.checkpointer = graph, checkpointer

    def _merge(self, state, update):
        """Apply each field's reducer — the declared merge rule replaces ad-hoc .append."""
        merged = dict(state)
        for key, value in update.items():
            reducer = self.g.reducers.get(key)
            merged[key] = reducer(state.get(key, []), ___) if reducer else value
        return merged

    def _next(self, node, state):
        """Routing: a conditional edge outranks a plain edge; no edge means END."""
        if node in self.g.branches:
            return self.g.branches[node](___)
        return self.g.edges.get(node, END)

    def invoke(self, input_state, config, verbose=True):
        thread_id = config["configurable"]["thread_id"]
        limit = config.get("recursion_limit", 25)
        ckpt = self.checkpointer.latest(thread_id) if self.checkpointer else None
        if ckpt is not None:                     # same thread_id -> resume, not restart
            state, node, step = ckpt["state"], ckpt["node"], ckpt["step"]
            if verbose:
                print(f"[resume] thread {thread_id!r} at step {step}, next node {node!r}")
        else:
            state, node, step = input_state, self._next(START, input_state), 0
        while node != ___ and step < limit:
            update = self.g.nodes[node](state)   # a node returns a DELTA ...
            state = self._merge(state, ___)   # ... the runtime merges it in
            step += 1
            node = self._next(node, state)
            if self.checkpointer:                # persist after EVERY node
                self.checkpointer.put(thread_id, step, node, state)
            if verbose:
                print(f"step {step:>2}: next node → {node}")
        return state


print("MiniGraph + CompiledGraph defined —",
      "the while-loop now lives inside a runtime, not in your agent code.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
class MiniGraph:
    """Mirror of LangGraph's StateGraph surface: nodes, edges, conditional edges, compile()."""

    def __init__(self, reducers):
        self.reducers = reducers                 # field name -> merge rule
        self.nodes, self.edges, self.branches = {}, {}, {}

    def add_node(self, name, fn):
        self.nodes[name] = fn

    def add_edge(self, src, dst):
        self.edges[src] = dst

    def add_conditional_edges(self, src, router):
        self.branches[src] = router

    def compile(self, checkpointer=None):
        return CompiledGraph(self, checkpointer)


class CompiledGraph:
    """The runtime: walks the graph, merges node updates via reducers, checkpoints
    after EVERY node, and resumes any thread_id that already has checkpoints."""

    def __init__(self, graph, checkpointer):
        self.g, self.checkpointer = graph, checkpointer

    def _merge(self, state, update):
        """Apply each field's reducer — the declared merge rule replaces ad-hoc .append."""
        merged = dict(state)
        for key, value in update.items():
            reducer = self.g.reducers.get(key)
            merged[key] = reducer(state.get(key, []), value) if reducer else value
        return merged

    def _next(self, node, state):
        """Routing: a conditional edge outranks a plain edge; no edge means END."""
        if node in self.g.branches:
            return self.g.branches[node](state)
        return self.g.edges.get(node, END)

    def invoke(self, input_state, config, verbose=True):
        thread_id = config["configurable"]["thread_id"]
        limit = config.get("recursion_limit", 25)
        ckpt = self.checkpointer.latest(thread_id) if self.checkpointer else None
        if ckpt is not None:                     # same thread_id -> resume, not restart
            state, node, step = ckpt["state"], ckpt["node"], ckpt["step"]
            if verbose:
                print(f"[resume] thread {thread_id!r} at step {step}, next node {node!r}")
        else:
            state, node, step = input_state, self._next(START, input_state), 0
        while node != END and step < limit:
            update = self.g.nodes[node](state)   # a node returns a DELTA ...
            state = self._merge(state, update)   # ... the runtime merges it in
            step += 1
            node = self._next(node, state)
            if self.checkpointer:                # persist after EVERY node
                self.checkpointer.put(thread_id, step, node, state)
            if verbose:
                print(f"step {step:>2}: next node → {node}")
        return state


print("MiniGraph + CompiledGraph defined —",
      "the while-loop now lives inside a runtime, not in your agent code.")
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- **Reducers**: `MiniGraph` is constructed with `{"messages": add_messages}` — every node
  returns only an *update* (a delta), and `_merge` applies the declared rule per field. This
  is the state-management service from the ledger in miniature: with parallel branches, a
  declared merge rule is what prevents two writers from clobbering each other.
- **Routing** (`_next`): a conditional edge (router function) outranks a plain edge; a node
  with neither goes to `END`. The plain loop's `if`-statement has become data plus one
  named, separately testable function.
- **Resume semantics**: each checkpoint row stores the state *and the next node to execute*.
  `invoke` first asks the checkpointer for the latest row of this `thread_id` — if one
  exists, the run continues there and `input_state` is ignored (that is why the kill test
  can pass `None`). A fresh `thread_id` starts from `START`.
- `recursion_limit` mirrors LangGraph's config key and is our step budget (S02: explicit
  stop conditions do not disappear inside frameworks — they just get renamed).
- `delete_after` exists for time travel (task R2) and clean test threads. Real LangGraph is
  gentler: it keeps the full history and *forks* from an earlier checkpoint instead of
  deleting.

</details>

> **Q:** Explain what a conditional edge in LangGraph is and which construct in the plain-code agent it replaces.
<details><summary>Click for answer</summary>

A conditional edge is a routing function attached to a node: after the node runs, the
function inspects the current state (typically the last message) and returns the name of the
next node. It replaces the implicit if-statement of the plain loop — "if the model emitted
tool calls, execute tools; otherwise stop". The difference is reification: the decision
becomes a named, separately testable function and part of an inspectable graph topology
instead of a branch buried in a while-loop.
</details>

In [ ]:
def agent_node(state):
    """One reasoning step. Note the reuse: Variant A's call_llm — and with it OUR retry
    policy — serves as the LLM port of Variant B too."""
    return {"messages": [call_llm(___)]}


graph = MiniGraph(reducers={"messages": add_messages})
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(TOOLS))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", ___)
graph.add_edge(___, "agent")          # the loop is an edge

app = graph.compile(checkpointer=SqliteSaver("data/runs.db"))
cfg = {"configurable": {"thread_id": "report-42"}, "recursion_limit": 25}

if OLLAMA_OK:
    app.checkpointer.delete_after("report-42", -1)     # clean thread for a fresh demo run
    final = app.invoke({"messages": init_messages(TASK)}, cfg)
    print("\nFINAL:", final["messages"][-1]["content"])
    print(f"checkpoints stored for thread 'report-42': "
          f"{len(app.checkpointer.history('report-42'))}")
else:
    print("Ollama not reachable — skipping the live run (the graph itself is fully built).")

<details>
<summary><b>Click here for the solution</b></summary>

```python
def agent_node(state):
    """One reasoning step. Note the reuse: Variant A's call_llm — and with it OUR retry
    policy — serves as the LLM port of Variant B too."""
    return {"messages": [call_llm(state["messages"])]}


graph = MiniGraph(reducers={"messages": add_messages})
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(TOOLS))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)
graph.add_edge("tools", "agent")          # the loop is an edge

app = graph.compile(checkpointer=SqliteSaver("data/runs.db"))
cfg = {"configurable": {"thread_id": "report-42"}, "recursion_limit": 25}

if OLLAMA_OK:
    app.checkpointer.delete_after("report-42", -1)     # clean thread for a fresh demo run
    final = app.invoke({"messages": init_messages(TASK)}, cfg)
    print("\nFINAL:", final["messages"][-1]["content"])
    print(f"checkpoints stored for thread 'report-42': "
          f"{len(app.checkpointer.history('report-42'))}")
else:
    print("Ollama not reachable — skipping the live run (the graph itself is fully built).")
```

</details>

> **Q:** What is a checkpointer in LangGraph and which three capabilities does it unlock?
<details><summary>Click for answer</summary>

A checkpointer persists the complete graph state to a backing store (e.g. SQLite or
Postgres) after every node execution, keyed by a thread identifier. It unlocks: (1) durable
resume — a crashed or killed run continues from the last checkpoint; (2) replay/time travel —
rewinding to an earlier checkpoint and re-executing, e.g. with a fixed prompt; (3)
human-in-the-loop — interrupting before a node and resuming hours later, since no live
process must wait.
</details>

## Part E — Measure, don't opine: the four measurements

The lecture's lab slide is explicit: *you measure rather than opine.* Four measurements:
**(1) the kill test** — kill the process mid-run, resume, verify the report comes out whole in
both variants; **(2) an injected bug** — a tool whose description quietly mismatches its
behaviour, and where you would look for it in each variant; **(3) lines of code**, counted
honestly per column; **(4) dependency count** — here the comparison is with what a *real*
framework install would pull in.

In [ ]:
def kill_test_plain():
    """Arm the crash, run Variant A until it dies, then resume from the JSON snapshot."""
    CRASH_BUDGET["calls_left"] = 2               # the 3rd LLM call kills the 'process'
    Path("run.json").unlink(missing_ok=True)
    try:
        run_agent(TASK, checkpoint="run.json", verbose=False)
        print("(run finished before the crash budget was spent)")
    except SimulatedCrash as exc:
        print("CRASH (plain):", exc)
    CRASH_BUDGET["calls_left"] = None            # disarm
    state = resume(___)                   # load the snapshot ...
    print(f"  resumed at step {state['step']} with {len(state['messages'])} messages")
    return run_agent(state=___, checkpoint="run.json", verbose=False)   # ... and continue


def kill_test_graph():
    """Same crash for Variant B — resuming is just invoking the same thread_id again."""
    CRASH_BUDGET["calls_left"] = 2
    kcfg = {"configurable": {"thread_id": "kill-test"}, "recursion_limit": 25}
    app.checkpointer.delete_after("kill-test", -1)          # start from a clean thread
    try:
        app.invoke({"messages": init_messages(TASK)}, kcfg, verbose=False)
        print("(run finished before the crash budget was spent)")
    except SimulatedCrash as exc:
        print("CRASH (graph):", exc)
    CRASH_BUDGET["calls_left"] = None
    return app.invoke(___, kcfg, verbose=False)   # same thread_id → resume from checkpoint


if OLLAMA_OK:
    sa = kill_test_plain()
    sb = kill_test_graph()
    ok_a = sa is not None and not sa["messages"][-1].get("tool_calls")
    ok_b = sb is not None and not sb["messages"][-1].get("tool_calls")
    print(f"\nkill test — resumed to a final answer:  Variant A: {ok_a}   Variant B: {ok_b}")
else:
    print("Ollama not reachable — skipping the kill test.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
def kill_test_plain():
    """Arm the crash, run Variant A until it dies, then resume from the JSON snapshot."""
    CRASH_BUDGET["calls_left"] = 2               # the 3rd LLM call kills the 'process'
    Path("run.json").unlink(missing_ok=True)
    try:
        run_agent(TASK, checkpoint="run.json", verbose=False)
        print("(run finished before the crash budget was spent)")
    except SimulatedCrash as exc:
        print("CRASH (plain):", exc)
    CRASH_BUDGET["calls_left"] = None            # disarm
    state = resume("run.json")                   # load the snapshot ...
    print(f"  resumed at step {state['step']} with {len(state['messages'])} messages")
    return run_agent(state=state, checkpoint="run.json", verbose=False)   # ... and continue


def kill_test_graph():
    """Same crash for Variant B — resuming is just invoking the same thread_id again."""
    CRASH_BUDGET["calls_left"] = 2
    kcfg = {"configurable": {"thread_id": "kill-test"}, "recursion_limit": 25}
    app.checkpointer.delete_after("kill-test", -1)          # start from a clean thread
    try:
        app.invoke({"messages": init_messages(TASK)}, kcfg, verbose=False)
        print("(run finished before the crash budget was spent)")
    except SimulatedCrash as exc:
        print("CRASH (graph):", exc)
    CRASH_BUDGET["calls_left"] = None
    return app.invoke(None, kcfg, verbose=False)   # same thread_id → resume from checkpoint


if OLLAMA_OK:
    sa = kill_test_plain()
    sb = kill_test_graph()
    ok_a = sa is not None and not sa["messages"][-1].get("tool_calls")
    ok_b = sb is not None and not sb["messages"][-1].get("tool_calls")
    print(f"\nkill test — resumed to a final answer:  Variant A: {ok_a}   Variant B: {ok_b}")
else:
    print("Ollama not reachable — skipping the kill test.")
```

</details>

In [ ]:
def web_search_buggy(query: str) -> str:
    """Search the saved research corpus. Returns the three most relevant sources."""
    # ^ the docstring — like the schema description the model reads — still PROMISES
    #   "most relevant". The implementation quietly delivers the opposite:
    terms = [t for t in query.lower().split() if len(t) > 2]
    ranked = sorted(corpus, key=lambda d: score(d, terms))   # ascending: LEAST relevant first
    return "\n\n".join(f"[{d['id']}] {d['title']} ({d['source_type']}, {d['date']})\n"
                       f"URL: {d['url']}\n{d['text'][:160]}…" for d in ranked[:3])


BUGGY_TOOLS = {"web_search": ___, "save_report": save_report}

probe_q = {"function": {"name": "web_search",
                        "arguments": {"query": "checkpointing and resume for agent frameworks"}}}
healthy = run_tool(probe_q)["content"]
buggy = run_tool(probe_q, ___)["content"]
print("--- healthy web_search, top hit ---\n" + healthy.split("\n\n")[0])
print("\n--- buggy web_search, top hit ---\n" + buggy.split("\n\n")[0])

# The same bug in Variant B enters only through the ToolNode binding — the wiring below is
# byte-for-byte the healthy topology. Nothing in the graph looks wrong.
buggy_graph = MiniGraph(reducers={"messages": add_messages})
buggy_graph.add_node("agent", agent_node)
buggy_graph.add_node("tools", ToolNode(___))
buggy_graph.add_edge(START, "agent")
buggy_graph.add_conditional_edges("agent", tools_condition)
buggy_graph.add_edge("tools", "agent")
buggy_app = buggy_graph.compile(checkpointer=SqliteSaver("data/runs.db"))
print("\nbuggy_app wired — same topology, quietly different behaviour.")
print("Where would you look first in each variant? Time yourself; the memo wants numbers.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
def web_search_buggy(query: str) -> str:
    """Search the saved research corpus. Returns the three most relevant sources."""
    # ^ the docstring — like the schema description the model reads — still PROMISES
    #   "most relevant". The implementation quietly delivers the opposite:
    terms = [t for t in query.lower().split() if len(t) > 2]
    ranked = sorted(corpus, key=lambda d: score(d, terms))   # ascending: LEAST relevant first
    return "\n\n".join(f"[{d['id']}] {d['title']} ({d['source_type']}, {d['date']})\n"
                       f"URL: {d['url']}\n{d['text'][:160]}…" for d in ranked[:3])


BUGGY_TOOLS = {"web_search": web_search_buggy, "save_report": save_report}

probe_q = {"function": {"name": "web_search",
                        "arguments": {"query": "checkpointing and resume for agent frameworks"}}}
healthy = run_tool(probe_q)["content"]
buggy = run_tool(probe_q, BUGGY_TOOLS)["content"]
print("--- healthy web_search, top hit ---\n" + healthy.split("\n\n")[0])
print("\n--- buggy web_search, top hit ---\n" + buggy.split("\n\n")[0])

# The same bug in Variant B enters only through the ToolNode binding — the wiring below is
# byte-for-byte the healthy topology. Nothing in the graph looks wrong.
buggy_graph = MiniGraph(reducers={"messages": add_messages})
buggy_graph.add_node("agent", agent_node)
buggy_graph.add_node("tools", ToolNode(BUGGY_TOOLS))
buggy_graph.add_edge(START, "agent")
buggy_graph.add_conditional_edges("agent", tools_condition)
buggy_graph.add_edge("tools", "agent")
buggy_app = buggy_graph.compile(checkpointer=SqliteSaver("data/runs.db"))
print("\nbuggy_app wired — same topology, quietly different behaviour.")
print("Where would you look first in each variant? Time yourself; the memo wants numbers.")
```

</details>

> **Q:** Why is the injected bug a tool whose *description* mismatches its *behaviour* — what does that choice test?
<details><summary>Click for answer</summary>

Such a bug lives at the model-tool boundary: the model reasons correctly from a wrong
description, so no exception fires and no stack trace points anywhere — diagnosis requires
inspecting what the model was told versus what the tool did. In plain code, both sides are in
your own ~100 lines. In the framework variant, tool wrapping and prompt assembly pass through
library layers, testing whether you can see through the abstraction. It operationalises the
opacity cost as a measurable time difference, rather than an opinion.
</details>

In [ ]:
def _source(obj):
    """inspect.getsource, with a fallback for CLASSES defined in notebook cells:
    on Python < 3.13 inspect cannot locate interactively defined classes, so we
    find the defining cell via a method's code object and slice the class block."""
    try:
        return inspect.getsource(obj)
    except (OSError, TypeError):
        import linecache
        code = next(c for c in (getattr(v, "__code__", None) for v in vars(obj).values())
                    if c is not None)
        lines = linecache.getlines(code.co_filename)         # the cell's cached source
        start = next(i for i, l in enumerate(lines)
                     if l.startswith(f"class {obj.__name__}"))
        block = [lines[start]]
        for line in lines[start + 1:]:
            if line.strip() and not line.startswith((" ", "\t")):
                break                                        # next top-level statement
            block.append(line)
        return "".join(block)


def loc(objs):
    """Count non-blank, non-comment source lines of the given functions/classes."""
    total = 0
    for obj in objs:
        for line in _source(obj).splitlines():
            s = line.strip()
            if s and not s.startswith(___):
                total += 1
    return total


shared_objs = [run_tool, init_messages, to_dict]        # both variants stand on these
variant_a_objs = [call_llm, save, resume, run_agent]    # the loop, retries, checkpointing
framework_objs = [add_messages, tools_condition, ToolNode, SqliteSaver,
                  MiniGraph, CompiledGraph]             # 'the dependency' — except we own it
wiring_objs = [agent_node]                              # plus ~8 wiring lines in Part D

ledger = pd.DataFrame([
    {"variant": "A — plain code",
     "agent LOC": loc(variant_a_objs) + loc(shared_objs),
     "framework LOC": 0,
     "extra deps": 0,
     "resume after crash": True,
     "history / replay": False,
     "who owns the loop": "you"},
    {"variant": "B — minigraph (LangGraph shape)",
     "agent LOC": loc(wiring_objs) + 8 + loc(shared_objs),
     "framework LOC": loc(___),
     "extra deps": 0,
     "resume after crash": True,
     "history / replay": True,
     "who owns the loop": "the runtime (here: still you)"},
])
ledger

<details>
<summary><b>Click here for the solution</b></summary>

```python
def _source(obj):
    """inspect.getsource, with a fallback for CLASSES defined in notebook cells:
    on Python < 3.13 inspect cannot locate interactively defined classes, so we
    find the defining cell via a method's code object and slice the class block."""
    try:
        return inspect.getsource(obj)
    except (OSError, TypeError):
        import linecache
        code = next(c for c in (getattr(v, "__code__", None) for v in vars(obj).values())
                    if c is not None)
        lines = linecache.getlines(code.co_filename)         # the cell's cached source
        start = next(i for i, l in enumerate(lines)
                     if l.startswith(f"class {obj.__name__}"))
        block = [lines[start]]
        for line in lines[start + 1:]:
            if line.strip() and not line.startswith((" ", "\t")):
                break                                        # next top-level statement
            block.append(line)
        return "".join(block)


def loc(objs):
    """Count non-blank, non-comment source lines of the given functions/classes."""
    total = 0
    for obj in objs:
        for line in _source(obj).splitlines():
            s = line.strip()
            if s and not s.startswith("#"):
                total += 1
    return total


shared_objs = [run_tool, init_messages, to_dict]        # both variants stand on these
variant_a_objs = [call_llm, save, resume, run_agent]    # the loop, retries, checkpointing
framework_objs = [add_messages, tools_condition, ToolNode, SqliteSaver,
                  MiniGraph, CompiledGraph]             # 'the dependency' — except we own it
wiring_objs = [agent_node]                              # plus ~8 wiring lines in Part D

ledger = pd.DataFrame([
    {"variant": "A — plain code",
     "agent LOC": loc(variant_a_objs) + loc(shared_objs),
     "framework LOC": 0,
     "extra deps": 0,
     "resume after crash": True,
     "history / replay": False,
     "who owns the loop": "you"},
    {"variant": "B — minigraph (LangGraph shape)",
     "agent LOC": loc(wiring_objs) + 8 + loc(shared_objs),
     "framework LOC": loc(framework_objs),
     "extra deps": 0,
     "resume after crash": True,
     "history / replay": True,
     "who owns the loop": "the runtime (here: still you)"},
])
ledger
```

</details>

**Reading the ledger — and the dependency count.** Variant B's *wiring* is tiny, but its
framework column is where the missing lines went: complexity is conserved, only its location
moves. On dependencies: both columns show 0 extra packages **because we mirrored the
framework instead of installing it**. For the real measurement the lecture asks for, run
`pip install langgraph` in a scratch virtualenv and count `pip freeze` — a real LangGraph
install pulls in on the order of **30+ transitive packages** (`langchain-core`, `pydantic`,
serializers, HTTP stacks, …), each with its own release schedule. That number — versus our
zero — *is* the lock-in and churn exposure the lecture priced, and it belongs in your memo.

> **Q:** Why can an agent's behaviour change after `pip install -U` even though no line of the developer's code changed? Name two defensive practices.
<details><summary>Click for answer</summary>

Because part of the token stream the model reads — embedded prompt templates,
tool-description formatting, stop heuristics — lives in the framework's code and changes with
its releases; the behavioural diff exists only in the dependency's repository. Defences: (1)
log the final rendered prompt (the literal tokens sent), so behaviour can be diffed across
versions; (2) pin dependencies and treat framework upgrades as program changes requiring
review and regression tests against an eval suite.
</details>

> **📝 Report task R1:** A colleague reads your Part E ledger and concludes: "the minigraph *wiring* is a fraction of the size of Variant A, so Variant B is the simpler system." Critique this inference using your **measured** LOC numbers: where exactly did the missing lines go, what does the LOC ratio actually measure, and under which circumstances is the relocation still a good deal?
> *No solution is provided — include your answer and a short justification in your lab report.*

### E4 — Time travel (report task)

Variant A's snapshot can only continue *from now*. The checkpointer's headline capability is
stronger: **rewind to step $k$ and replay**. Implement it on our own `SqliteSaver`.

> **📝 Report task R2 (code):** Complete the cell below — implement `rewind`, the **time-travel/replay** feature the lecture called the checkpointer's headline trick ("rewind to step 6 and replay"): drop every checkpoint of a thread after step `k`, then re-invoke the thread so it continues from that snapshot. Then answer in your report: with `temperature > 0`, is the replayed continuation guaranteed to be identical to the original run? What exactly *is* reproduced exactly?
> *No solution is provided — include your code and a short justification in your lab report.*

In [ ]:
def rewind(thread_id, to_step, limit=25):
    """Time travel: drop every checkpoint of `thread_id` after `to_step`, then re-invoke
    the thread — it resumes from that snapshot ('rewind to step k and replay')."""
    hist = app.checkpointer.history(thread_id)
    if not hist:
        print(f"No checkpoints for thread {thread_id!r} — run Part D first.")
        return None
    print(f"thread {thread_id!r}: {len(hist)} checkpoints → rewinding to step {to_step}")
    app.checkpointer.delete_after(___, ___)
    cfg2 = {"configurable": {"thread_id": ___}, "recursion_limit": limit}
    return app.invoke(None, cfg2)


if OLLAMA_OK:
    replayed = rewind("report-42", to_step=2)
    if replayed is not None:
        print("\nREPLAYED FINAL:", replayed["messages"][-1]["content"][:300])
else:
    print("Ollama not reachable — skipping the replay.")

> **📝 Report task R3:** Write the one-page memo the lecture announced: addressed to the CTO of a fictional two-person startup that must ship the research agent in three months and maintain it for at least two years. Recommend **Variant A** (plain code), **Variant B** (a graph framework), or a staged path between them. Defend the recommendation with the six-row decision checklist (team, lifetime, control flow, observability, persistence, maintenance) and use your Part E measurements — LOC, dependency count, the injected-bug diagnosis, the kill test — as evidence, not vibes.
> *No solution is provided — attach the memo to your lab report.*

> **📝 Report task R4:** Audit this lab's Variant B against the lecture's four exit-strategy practices — **ports and adapters**, **own every prompt**, **own the message log**, **evals as a safety net**. Which practices does the notebook's setup already implement (point to the concrete cell, object or design decision), which are missing, and what would each missing one cost to add?
> *No solution is provided — include the audit in your lab report.*

## Part F — Tuning & exploration

*No gaps in this part — everything runs as given.* Knobs worth twisting, each of which feeds
an argument in your memo:

- **`recursion_limit` vs `MAX_STEPS`** — shrink them until the agent stops finishing its
  report; note that the stop condition survived the framework move, merely renamed.
- **`temperature`** (via `OPTIONS`) — rerun the R2 replay at `0.0` and at `0.8`: when is a
  replayed continuation reproducible?
- **retry patience** — change `retries` and the backoff base in `call_llm` and reason about
  worst-case wait ($\sum_k 2^k$ seconds).
- **the research question** — edit the `question` argument; does the agent pick different
  sources from the corpus?
- **`thread_id`** — every fresh value opens an independent, resumable run; inspect
  `app.checkpointer.history(...)` for any of them.

In [ ]:
def tuned_run(question=TASK, thread_id="tune-1", temperature=0.2, recursion_limit=25):
    """One fresh Variant-B run with the given knobs; prints a compact summary."""
    if not OLLAMA_OK:
        print("Ollama not reachable — tuning needs a live model.")
        return None
    OPTIONS["temperature"] = temperature
    app.checkpointer.delete_after(thread_id, -1)          # fresh thread
    tcfg = {"configurable": {"thread_id": thread_id}, "recursion_limit": recursion_limit}
    t0 = time.time()
    final = app.invoke({"messages": init_messages(question)}, tcfg, verbose=False)
    hist = app.checkpointer.history(thread_id)
    print(f"thread {thread_id!r}: {len(hist)} checkpoints, "
          f"{len(final['messages'])} messages, {time.time() - t0:.1f} s")
    print("FINAL:", final["messages"][-1]["content"][:200])
    return final


# State growth across the checkpoints of the Part D run — persistence made visible.
hist = app.checkpointer.history("report-42")
if hist:
    plt.figure(figsize=(5.5, 3))
    plt.plot([h["step"] for h in hist],
             [len(h["state"]["messages"]) for h in hist], marker="o")
    plt.xlabel("checkpoint step")
    plt.ylabel("messages in state")
    plt.title("Every node leaves a checkpoint")
    plt.tight_layout()
    plt.show()
else:
    print("(no checkpoints for 'report-42' yet — run Part D with Ollama available)")

# Optional slider UI; plain function calls work just as well.
try:
    import ipywidgets as widgets
    from IPython.display import display
    ui = widgets.interact_manual(tuned_run,
                                 question=widgets.fixed(TASK),
                                 thread_id=widgets.fixed("tune-widget"),
                                 temperature=(0.0, 1.0, 0.1),
                                 recursion_limit=(6, 40, 1))
    print("Set the sliders, then press 'Run Interact' (each run takes a while).")
except Exception:
    print("ipywidgets not installed — call tuned_run(...) directly, e.g.:")
    print("  tuned_run(temperature=0.8, thread_id='tune-hot', recursion_limit=16)")

## Wrap-up

**Takeaways**

- Five families, one loop: LangGraph (state graph), CrewAI (role team), AutoGen
  (conversation), smolagents (code actions), vendor SDKs (packaged loop) — they differ in
  what they make explicit, not in what they fundamentally do.
- The plain baseline is small and honest: a dict registry, an eight-line retry wrapper, a
  two-line JSON checkpoint saved after every step. Its gaps — history/replay, threads,
  suspension — are precisely what a checkpointer sells.
- You rebuilt LangGraph's API shape in ~60 owned lines: reducers made the merge rule
  explicit, a conditional edge reified the if-statement, *the loop became an edge*, and
  resume became a `thread_id` detail. Complexity was conserved; ownership moved — and this
  week it moved somewhere you can read.
- Decide with the six-row checklist, price the **timing asymmetry** (benefits at the demo,
  costs in month three), and whatever you adopt: **design for divorce** — own your prompts,
  own the message log, hide frameworks behind adapters, keep an eval suite as the exit door.

**Next week:** Session 08 — *Memory*: the context window is not memory; short-term context,
long-term stores, and what an agent should remember at all.

---

### 📝 For your lab report

| Task | What to hand in |
|---|---|
| **R1** | Critique of the "fewer lines = simpler system" inference, argued with your measured LOC ledger |
| **R2** | Your completed `rewind` code **plus** the answer: what exactly does replay reproduce when `temperature > 0`? |
| **R3** | The one-page CTO memo: recommendation defended with the six-row decision checklist and your Part E measurements (LOC, dependencies, bug diagnosis, kill test) |
| **R4** | The design-for-divorce audit of Variant B: which of the four exit practices are implemented / missing, with pointers into the notebook |

*Reminder: report tasks have no solutions in this notebook — your own reasoning is the deliverable.*